# Interactive Single-Host TorchTitan Training on Cloud TPU Workspaces

This notebook demonstrates how to run **TorchTitan** (`torchtitan.experiments.tpu.train`) on a single-host **GKE Cloud TPU Workspace** (`tpu_1` with 1 TPU chip or `tpu_4` with 4 TPU chips), connected directly from local VS Code via **`jupyter-workspace-sync`** (`vscode-extension`) or in-browser JupyterLab.

### How Workspaces + `jupyter-workspace-sync` Simplifies Single-Host TorchTitan

| Step | Legacy TPU VM Process (`torchtitan/experiments/tpu/README.md`) | GKE Workspaces + `jupyter-workspace-sync` |
| :--- | :--- | :--- |
| **1. Provisioning & Access** | Manual `gcloud compute tpus tpu-vm create` + SSH key management (`gcloud compute tpus tpu-vm ssh`) | **1-click Workspace creation** in UI (`TPU v5 1x1` or `TPU v5 2x2`), zero SSH keys (IAP + K8s RBAC) |
| **2. `torch_tpu` Installation** | Install Python 3.12, clone `torch_tpu`, edit `pyproject.toml`, run `bazel build -c opt //ci/wheel:torch_tpu_wheel` (**~30–45 min**) | **0 min**: `torch>=2.12`, `torch_tpu`, `jax[tpu]`, and `libtpu==0.0.48` are pre-installed in `jupyterlab:latest-tpu` |
| **3. Code & Config Editing** | Manual `git clone` / `scp` over SSH or editing in terminal | **~300 ms live sync** from local VS Code on every `Ctrl+S` save via `jupyter-workspace-sync` |
| **4. Shared Environment** | Isolated to a single GCE VM; must rebuild a Docker image later to run multi-host | Installs pure-Python TorchTitan deps once into `/home/jovyan/shared/.pydeps` (`ReadWriteMany` PVC), shared automatically with multi-host GKE jobs |

## 1. Hardware Pre-Flight Check & Install TorchTitan Requirements

Assuming you cloned your repositories locally (e.g., `google-pytorch/torchtitan`) and synced your root project to the Workspace via `jupyter-sync`, we:
1. Locate the synced `torchtitan` repository on the Workspace filesystem.
2. Run `tpu_lock.preflight_check()` to ensure no lingering processes hold `/dev/vfio/*` TPU locks.
3. Install TorchTitan's dependencies directly from the repository's own requirement files (`torchtitan/experiments/tpu/requirements.txt`, `requirements.txt`, and `.ci/docker/requirements-flux.txt`) into `/home/jovyan/shared/.pydeps` (`PYTHONUSERBASE`), so they persist across restarts and are shared with multi-host worker pods.

In [ ]:
import importlib
import importlib.util
import os
import pathlib
import subprocess
import sys

# Ensure generic TPU helper modules (tpu_lock, tpu_trainer) are importable
for candidate in [pathlib.Path.cwd(), pathlib.Path("/home/jovyan/shared/gke-workspaces/examples/torchtitan")]:
    if (candidate / "tpu_trainer.py").exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

import tpu_lock
import tpu_trainer

importlib.reload(tpu_lock)
importlib.reload(tpu_trainer)

# 1. Verify TPU device nodes are unlocked and ready for torchrun
assert tpu_lock.preflight_check(is_distributed=True, auto_clear=True), "TPU hardware is locked!"

# 2. Locate the synced TorchTitan repository (e.g., synced from google-pytorch/torchtitan via jupyter-sync)
search_paths = [
    pathlib.Path(os.environ["TORCHTITAN_DIR"]) if "TORCHTITAN_DIR" in os.environ else None,
    pathlib.Path.cwd() / "torchtitan",
    pathlib.Path.cwd().parent / "torchtitan",
    pathlib.Path("/home/jovyan/shared/google-pytorch/torchtitan"),
    pathlib.Path("/home/jovyan/google-pytorch/torchtitan"),
    pathlib.Path("/home/jovyan/shared/torchtitan"),
    pathlib.Path("/home/jovyan/torchtitan"),
]
torchtitan_dir = next(
    (p.resolve() for p in search_paths if p and (p / "torchtitan/experiments/tpu/train.py").exists()),
    None,
)
if torchtitan_dir is None:
    raise FileNotFoundError(
        "Could not find synced 'torchtitan' repository. Sync your root project "
        "(e.g. google-pytorch) via `jupyter-sync sync` or set TORCHTITAN_DIR."
    )

# 3. Install TorchTitan's requirements from the repository's requirements files into shared PYTHONUSERBASE
user_base = pathlib.Path("/home/jovyan/shared/.pydeps")
user_base.mkdir(parents=True, exist_ok=True)
tpu_trainer.configure_runtime_env(user_base=user_base, extra_pythonpath=torchtitan_dir)

required_mods = ("tyro", "grain", "tokamax", "spmd_types", "attn_gym", "torch_remat", "torchao", "transformers")
missing_mods = [m for m in required_mods if importlib.util.find_spec(m) is None]
if missing_mods:
    print(f"Installing TorchTitan requirements ({missing_mods}) from {torchtitan_dir} into {user_base}...")
    pip_env = {**os.environ, "PYTHONUSERBASE": str(user_base)}
    base_req_file = (
        torchtitan_dir / "requirements.txt"
        if (torchtitan_dir / "requirements.txt").exists()
        else torchtitan_dir / ".ci/docker/requirements.txt"
    )
    # Skip jax/jaxlib/libtpu lines since the TPU runtime is already pre-installed in the Workspace image
    tpu_reqs = [
        line.strip()
        for line in (torchtitan_dir / "torchtitan/experiments/tpu/requirements.txt").read_text().splitlines()
        if line.strip() and not line.strip().startswith(("#", "jax==", "jaxlib==", "libtpu=="))
    ]
    for cmd in [
        [sys.executable, "-m", "pip", "install", "--user", "-q", *tpu_reqs],
        [
            sys.executable, "-m", "pip", "install", "--user", "-q",
            "-r", str(base_req_file),
            "-r", str(torchtitan_dir / ".ci/docker/requirements-flux.txt"),
        ],
        [
            sys.executable, "-m", "pip", "install", "--user", "-q", "--no-deps",
            "torchao==0.17.0", "typeguard>=4.0.0",
        ],
    ]:
        subprocess.check_call(cmd, env=pip_env)
    tpu_trainer.configure_runtime_env(user_base=user_base, extra_pythonpath=torchtitan_dir)
    print("TorchTitan dependencies installed.")
else:
    print(f"All TorchTitan dependencies are ready in '{user_base}'.")

tpu_devices = tpu_lock.get_tpu_devices()
print(f"\nWorkspace Pod          : {os.environ.get('HOSTNAME', 'local')}")
print(f"TPU Accelerator Type   : {os.environ.get('TPU_ACCELERATOR_TYPE', 'unknown')}")
print(f"Attached TPU Chips     : {len(tpu_devices)} ({tpu_devices})")
print(f"Synced TorchTitan Repo : {torchtitan_dir}")
print(f"Shared Python Userbase : {user_base}")

## 2. Run Single-Host TorchTitan Smoke Test (`llama3_debugmodel`)

We first run the lightweight `llama3_debugmodel` recipe from `torchtitan.experiments.tpu.llama3` across the attached TPU chip(s) using `run_singlehost_torchtitan()`.

Under the hood, `run_singlehost_torchtitan()`:
1. Auto-detects `nproc_per_node` (`1` on a `tpu_1` workspace or `4` on a `tpu_4` workspace).
2. Populates `TORCH_TPU_SLICEBUILDER_ADDRESSES` and `TORCH_TPU_TOPOLOGY` via `torch.tpu.distributed.environment`.
3. Executes `torchrun -m torchtitan.experiments.tpu.train` with FSDP2 (`bfloat16`) using the built-in `c4_test` dataset and test tokenizer in `tests/assets/`.

In [ ]:
def make_torchtitan_log_filter():
    """Returns a log line filter that surfaces TorchTitan rank-0 step metrics and errors."""
    keywords = (
        "Device type:",
        "Initializing TPU",
        "[titan]",
        "Traceback",
        "RuntimeError",
        "ValueError",
        "Error:",
    )
    return lambda line: any(k in line for k in keywords)


tpu_trainer.run_singlehost_training(
    main_script="torchtitan.experiments.tpu.train",
    module=True,
    src_dir=torchtitan_dir,
    user_base=user_base,
    args=[
        "--module=torchtitan.experiments.tpu.llama3",
        "--config=llama3_debugmodel",
        "--training.steps=5",
        "--training.max_context_length=256",
        "--metrics.log_freq=1",
        "--metrics.enable_tensorboard",
    ],
    log_filter=make_torchtitan_log_filter(),
)

## 3. Train Llama 3.2 1B (`llama3_1b`) with FSDP2 & TensorBoard Logging

Next, we run the full **Llama 3.2 1B** (`llama3_1b`) model configuration on our TPU workspace with `bfloat16` precision, selective activation checkpointing (`SelectiveAC`), and TensorBoard metrics export enabled (`--metrics.enable_tensorboard`).

Whenever you edit any file in your local `torchtitan` repository in VS Code (`Ctrl+S`), `jupyter-workspace-sync` pushes the change to `/home/jovyan/shared/torchtitan` in ~300 ms—so re-running this cell immediately executes your updated model or trainer code on the TPU!

In [ ]:
tpu_trainer.run_singlehost_training(
    main_script="torchtitan.experiments.tpu.train",
    module=True,
    src_dir=torchtitan_dir,
    user_base=user_base,
    args=[
        "--module=torchtitan.experiments.tpu.llama3",
        "--config=llama3_1b",
        "--training.steps=5",
        "--training.max_context_length=128",
        "--training.dtype=bfloat16",
        "--training.mixed_precision_param=bfloat16",
        "--metrics.log_freq=1",
        "--metrics.enable_tensorboard",
    ],
    log_filter=make_torchtitan_log_filter(),
)

## 4. Verify TPU Lock Release & Inspect TensorBoard Outputs

When `torchrun` finishes, its child worker process exits and releases `/dev/vfio/*`. Because our outputs directory resides inside `/home/jovyan/shared/torchtitan/outputs`, TensorBoard event files are immediately accessible from the Workspace pod.

In [ ]:
# Verify that all TPU device nodes are unlocked
locked = tpu_lock.check_tpu_locks(verbose=True)
assert not locked, f"Expected TPU devices to be free, found: {locked}"

# Inspect TensorBoard event files generated in /home/jovyan/shared/torchtitan/outputs/tb
tb_dir = torchtitan_dir / "outputs" / "tb"
if tb_dir.exists():
    event_files = sorted(tb_dir.rglob("events.out.tfevents.*"))
    print(f"\n📊 Found {len(event_files)} TensorBoard event file(s) under {tb_dir}:")
    for ef in event_files[-5:]:
        print(f"   • {ef.relative_to(torchtitan_dir)} ({ef.stat().st_size} bytes)")